In [8]:
import numpy as np
import sys, os, time
import torch
import torch.nn as nn
import optuna
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import matplotlib as mpl
from torch.utils.data import random_split, DataLoader

mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'serif'


# import data_combined, architecture
import data, architecture

In [9]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch

# Test it
print(f"Current Device: {torch.cuda.current_device()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}") 
# Note: Because of the environment variable, GPU 1 now looks like '0' to PyTorch

Current Device: 0
Device Name: NVIDIA A100 80GB PCIe


# Get top 10 models - combined, revisions OR FT 0 BSQ 
training set = n BSQ sims + n sims
test and valid set = just 300 n sims

In [10]:
################################### INPUT ############################################
# data parameters

f_Pk_norm = None #file with Pk to normalize Pk
seed      = 42       #seed to split data in train/valid/test
mode      = 'test'   #'train','valid','test' or 'all'

# params_ext = '_lcdm'

Pk_type = 'MPk'
cosm_type = 'nwLH'
cosm_type_data_range = 'nwLH'
k_max = 0.3

model_type = 'dynamic' #or 'dynamic' 'dynamic_fixed_final'
log = True

output_size = 6
if k_max == 0.5:
    input_size = 79
elif k_max == 0.3:
    input_size = 47
elif k_max == 0.1:
    input_size = 15

additional_extension =  '_k_'+str(k_max)    #'_combined_train_val_revisions'   #FIXED TEST/VALID SET TO 300
params_ext = ''

n_sims_cosm = [50, 100, 200, 400, 800, 1400]
n_sims_BSQ = [0]


In [11]:
for n_BSQ in n_sims_BSQ: 
    for n_sims in n_sims_cosm:
        n = []
        mse = []

        name = 'transfer10_network2_'+str(n_sims)+'_'+cosm_type+'_'+str(n_BSQ)+'_BSQ' + additional_extension

                
        if additional_extension == '_combined_train_val_revisions':
            if mode == 'test':
                f_Pk     = 'Pk_files/'+cosm_type+'_'+Pk_type+'_'+mode+'.npy'
                f_params  = 'real_params/'+cosm_type+'_'+Pk_type+'_params_'+mode+'.txt'
    
            else:
                f_Pk      = 'Pk_files/'+Pk_type+'_'+ str(n_sims) +'_'+ cosm_type +'_'+ str(n_BSQ)+'_BSQ_combined_'+mode+'.npy'
                f_params  =    'real_params/'+Pk_type+'_'+str(n_sims)+'_'+ cosm_type+'_'+ str(n_BSQ)+'_BSQ_combined_params_'+mode+'.txt'

        else:
            f_Pk      = 'Pk_files/'+'all_'+str(Pk_type)+'_'+str(cosm_type)+params_ext+additional_extension+'.npy'
            f_params  = 'real_params/'+'all_' + str(cosm_type)+'_params'+params_ext+'.txt' 
            
        
        # input_size  = 79
        # log = True

        study_name = str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)
        mother = '/scratch/network/vk9342/USRP2024_scratch/pytorch/'+str(Pk_type)+'_'+str(cosm_type)+'/'+str(name)+'/models/'

        print(name)
        print(f_params)
        print(f_Pk)
        
    
        # architecture parameters
        # training parameters
        batch_size = 32
        # optuna parameters
        storage    = 'sqlite:///nwLH.db'
        
        
        
        ######################################################################################
        ######################################################################################
        # use GPUs if available
        if torch.cuda.is_available():
            print("CUDA Available")
            device = torch.device('cuda')
        else:
            print('CUDA Not Available')
            device = torch.device('cpu')
        
        # load the optuna study
        study = optuna.load_study(study_name=study_name, storage=storage)
        
        # get the scores of the study trials
        values = np.zeros(len(study.trials))
        completed = 0
        
        for i,t in enumerate(study.trials):
            values[i] = t.value
            if t.value is not None:  completed += 1
        
        # get the info of the best trial
        indexes = np.argsort(values)
        for indx in range(10):
            trial = study.trials[indexes[indx]]
        
            # print("\nTrial number {}".format(trial.number))
            # print("Value: %.5e"%trial.value)
            for key, value in trial.params.items():
                lr       = trial.params['lr']
                wd       = trial.params['wd']
                n_layers       = trial.params['n_layers']
                p       = trial.params['dropout_l']
                
            fmodel = mother +'model_%d.pt'%trial.number

            
        
            # # generate the architecture
            if model_type == 'dynamic_fixed_final':
                out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers-1)]
                model = architecture.dynamic_model_fixed_final(trial, input_size, output_size, 
                                                               final_hidden_layer_size, 
                                                               n_layers, p, out_fs,
                                                               max_neurons_layers=500)
                
            elif model_type == 'dynamic':
                out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers)]
                model = architecture.dynamic_model(trial, input_size, output_size,
                                                   n_layers, p, out_fs,
                                                   max_neurons_layers=500)   

            else:
                print('not a valid model')

            # print('N BSQ: ', str(n_BSQ), 'N nwLH: ', str(n_nwLH), 'layer sizes: ', out_fs)
            # print('N BSQ: ', str(n_BSQ), 'N nwLH: ', str(n_nwLH), 'n layers: ', n_layers)
    
            
            model.to(device)  
            
            # load best-model, if it exists
            if os.path.exists(fmodel):  
                print('Loading model...')
                model.load_state_dict(torch.load(fmodel, map_location=torch.device(device)))
            else:
                raise Exception('model doesnt exists!!!')
            
            
            # define loss function
            criterion = nn.MSELoss() 
            
            # get the data
            if additional_extension == '_combined_train_val_revisions':
                test_loader = data_combined.create_dataset(mode, seed, f_Pk, f_Pk_norm, f_params, 
                                                  batch_size, shuffle=False, workers=1, 
                                                  cosm_type =cosm_type, log=log) 
            else:
                test_loader = data.create_dataset(mode, seed, f_Pk, f_Pk_norm, f_params, 
                                              batch_size, shuffle=False, workers=1, 
                                              cosm_type =cosm_type, log=log, shuffle_all = True) 
                                              
            test_points = 0
            for x,y in test_loader:  test_points += x.shape[0]
            
            # define the arrays containing the true and predicted value of the parameters
            params  = output_size
            results = np.zeros((test_points, 2*params), dtype=np.float32)
            
            # test the model
            test_loss, points = 0.0, 0
            model.eval()
            with torch.no_grad():
                for x, y in test_loader:
                    bs   = x.shape[0]  #batch size
                    x, y = x.to(device), y.to(device)
                    y_NN = model(x)
                    test_loss += (criterion(y_NN, y).item())*bs
                    results[points:points+bs,0*params:1*params] = y.cpu().numpy()
                    results[points:points+bs,1*params:2*params] = y_NN.cpu().numpy()
                    points    += bs
            test_loss /= points
            print('Test loss:', test_loss)
            
            fout = '../results/Results_'+mode+'_'+Pk_type+'_'+cosm_type+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt'
            np.savetxt(fout, results)


transfer10_network2_50_nwLH_0_BSQ_k_0.3
real_params/all_nwLH_params.txt
Pk_files/all_MPk_nwLH_k_0.3.npy
CUDA Available
Loading model...
Test loss: 0.03606810877720515
Loading model...
Test loss: 0.036309116284052534
Loading model...
Test loss: 0.03700670863191287
Loading model...
Test loss: 0.03703440015514692
Loading model...
Test loss: 0.03674641718467077
Loading model...
Test loss: 0.03647776812314987
Loading model...
Test loss: 0.03615241423249245
Loading model...
Test loss: 0.03653091768423716
Loading model...
Test loss: 0.03670923247933388
Loading model...
Test loss: 0.03722509895761808
transfer10_network2_100_nwLH_0_BSQ_k_0.3
real_params/all_nwLH_params.txt
Pk_files/all_MPk_nwLH_k_0.3.npy
CUDA Available
Loading model...
Test loss: 0.031193664669990538
Loading model...
Test loss: 0.031052663226922354
Loading model...
Test loss: 0.030918385883172354
Loading model...
Test loss: 0.031759458531936006
Loading model...
Test loss: 0.03229460562268893
Loading model...
Test loss: 0.030825

# Get BSQ pretrain models (with node)

In [46]:
################################### INPUT ############################################
# data parameters
f_Pk_norm = None #file with Pk to normalize Pk
seed      = 42       #seed to split data in train/valid/test
mode      = 'train'   #'train','valid','test' or 'all'

cosm_type = 'BSQ' #'nwLH' or 'LH' or 'LC'
Pk_type = 'Pk'
k_max = 0.3

additional_extension = '_k_'+str(k_max)   #'_node_v2_fR'
params_ext = ''

model_type = 'dynamic' #or 'dynamic' 'dynamic_fixed_final'
output_size = 6
log = True

all_n_sims_BSQ = [2000, 22000]


In [47]:
for n_sims_BSQ in all_n_sims_BSQ:
 
    additional_extension = '_k_'+str(k_max)   #'_node_v2_fR'
    f_Pk      = 'Pk_files/'+'all_'+str(Pk_type)+'_'+str(cosm_type)+params_ext+additional_extension+'.npy'
    f_params  = 'real_params/' + 'all_'+str(cosm_type)+'_params'+params_ext+'.txt' 

    name = 'transfer10_network1_'+str(n_sims_BSQ)+'_BSQ' + additional_extension

    if k_max == 0.5:
        input_size = 79
    elif k_max == 0.3:
        input_size = 47
    elif k_max == 0.1:
        input_size = 15


    # INFO THAT DOES NOT CHANGE
    study_name  = str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)   
    mother = '/scratch/network/vk9342/USRP2024_scratch/pytorch/'+str(Pk_type)+'_'+str(cosm_type)+'/'+str(name)+'/models/'
    
    print(name)
    print(f_params)
    print(f_Pk)
    print(study_name)
    
    # architecture parameters
    # training parameters
    batch_size = 32
    # optuna parameters
    storage    = 'sqlite:///nwLH.db'
    
    ######################################################################################
    ## LOAD MODEL
    
    # use GPUs if available
    if torch.cuda.is_available():
        print("CUDA Available")
        device = torch.device('cuda')
    else:
        print('CUDA Not Available')
        device = torch.device('cpu')
        
    ########################################################################
    
    
    # load the optuna study
    study = optuna.load_study(study_name=study_name, storage=storage)
    # get the scores of the study trials
    values = np.zeros(len(study.trials))
    completed = 0
    for i,t in enumerate(study.trials):
        values[i] = t.value
        if t.value is not None:  completed += 1

    # get the info of the best trial            
    indexes = np.argsort(values)
    for indx in range(10):  #choose the best-model here, e.g. [0], or [1]
        trial = study.trials[indexes[indx]]
        print("\nTrial number {}".format(trial.number))
        print("Value: %.5e"%trial.value)
        print(" Params: ")
        for key, value in trial.params.items():
            print("    {}: {}".format(key, value))
        lr       = trial.params['lr']
        wd       = trial.params['wd']
        n_layers       = trial.params['n_layers']
        p       = trial.params['dropout_l']
        fmodel = mother +'model_%d.pt'%trial.number
    
        ## generate the architecture
        # out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers-1)]
       # generate architecture
        if model_type == 'dynamic_fixed_final':
            model = architecture.dynamic_model_fixed_final(trial, input_size, 
                                                           output_size, final_hidden_layer_size,
                                                           n_layers, p, out_fs, max_neurons_layers=500
                                                          ).to(device)
        elif model_type == 'dynamic':
            out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers)]
            model = architecture.dynamic_model(trial, input_size, 
                                               output_size,
                                               n_layers, p, out_fs,
                                               max_neurons_layers=500
                                              ).to(device) 

        
        model.to(device)  
        # load best-model, if it exists
        if os.path.exists(fmodel):  
            print('Loading model...')
            model.load_state_dict(torch.load(fmodel, map_location=torch.device(device)))
        else:
            raise Exception('model doesnt exists!!!')
        
        # define loss function
        criterion = nn.MSELoss() 


        ## GET TRAIN, VALID, TEST RESULTS
        # for mode in ['test']:
        # get the data
        if mode == 'train':
                # get the data
                full_train_dataset = data.create_dataset('train', seed, f_Pk, f_Pk_norm, 
                                                   f_params, batch_size, shuffle=False, 
                                                   workers=1, cosm_type = cosm_type, log=log, 
                                                   shuffle_all=True).dataset
                train_subset, _ = torch.utils.data.random_split(full_train_dataset,
                                                                [n_sims_BSQ, len(full_train_dataset) - n_sims_BSQ],
                                                                generator=torch.Generator().manual_seed(seed)
                                                               )
                test_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=1)
        else:
            test_loader = data.create_dataset(mode, seed, f_Pk, f_Pk_norm, f_params, 
                                          batch_size, shuffle=False, workers=1, 
                                          cosm_type =cosm_type, log=log, 
                                          shuffle_all = True)
        
        
        test_points = 0
        for x,y in test_loader:  test_points += x.shape[0]
        
        # define the arrays containing the true and predicted value of the parameters
        params  = 5 #output_size
        results = np.zeros((test_points, 2*params), dtype=np.float32)
        
        # test the model
        test_loss, points = 0.0, 0
        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                bs   = x.shape[0]  #batch size
                x, y = x.to(device), y.to(device)
                y_NN = model(x)
                test_loss += (criterion(y_NN[:, :5], y).item())*bs
                results[points:points+bs,0*params:1*params] = y.cpu().numpy()
                results[points:points+bs,1*params:2*params] = y_NN[:, :5].cpu().numpy()
                points    += bs
        test_loss /= points
        print('Test loss:', test_loss)
        
        # save results to file
        # fout = 'results/Results_'+mode+'_'+str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt'
        
        fout = '../results/Results_'+mode+'_'+Pk_type+'_'+cosm_type+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt'
        np.savetxt(fout, results)
        print('results/Results_'+mode+'_'+str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt')
    
    
    
                    

transfer10_network1_2000_BSQ_k_0.3
real_params/all_BSQ_params.txt
Pk_files/all_Pk_BSQ_k_0.3.npy
Pk_BSQ_params_transfer10_network1_2000_BSQ_k_0.3
CUDA Available

Trial number 32
Value: 2.00159e-02
 Params: 
    lr: 0.00033909899546953284
    wd: 2.7670202764128798e-05
    n_layers: 2
    dropout_l: 0.3695294698731181
    n_units_l0: 455
    n_units_l1: 230
Loading model...
Test loss: 0.014936507850885392
results/Results_train_Pk_BSQ_params_transfer10_network1_2000_BSQ_k_0.3_model0.txt

Trial number 71
Value: 2.01597e-02
 Params: 
    lr: 0.0007284988574330813
    wd: 0.006415701926937146
    n_layers: 2
    dropout_l: 0.3144114248260245
    n_units_l0: 483
    n_units_l1: 387
Loading model...
Test loss: 0.015108023054897786
results/Results_train_Pk_BSQ_params_transfer10_network1_2000_BSQ_k_0.3_model1.txt

Trial number 65
Value: 2.02145e-02
 Params: 
    lr: 0.0007766725078372029
    wd: 0.02145389899949189
    n_layers: 2
    dropout_l: 0.3176248792858066
    n_units_l0: 485
    n_units

# Get fine tune models (node) --> revisions with diff k cuts

In [3]:
################################### INPUT ############################################
# data parameters
f_Pk_norm = None #file with Pk to normalize Pk
seed      = 42       #seed to split data in train/valid/test
mode      = 'test'   #'train','valid','test' or 'all'

cosm_type = 'nwLH'
Pk_type = 'MPk'
k_max = 0.3

additional_extension = '_k_'+str(k_max)     #'_fixed_ft_test'   #FIXED TEST/VALID SET TO 300
params_ext = ''

model_type = 'dynamic' #or 'dynamic' 'dynamic_fixed_final'
output_size = 6

all_n_sims = [50, 100, 200, 400, 800, 1400]
all_n_sims_BSQ = [2000, 22000]

log = True

if k_max == 0.5:
    input_size = 79
elif k_max == 0.3:
    input_size = 47
elif k_max == 0.1:
    input_size = 15
        
final_hidden_layer_size = 10

In [4]:
for n_sims_BSQ in all_n_sims_BSQ:
    for n_sims in all_n_sims:
        # NAME (depending on network 1 or 2)
        name = 'transfer10_network2_'+str(n_sims)+'_'+cosm_type+'_'+str(n_sims_BSQ)+'_BSQ' + additional_extension  #_fixed for fixed test/valid set
        
        # OTHER EXTENSIONS
        # params_ext = ''  
        f_Pk      = 'Pk_files/'+'all_'+str(Pk_type)+'_'+str(cosm_type)+params_ext+additional_extension+'.npy'
        f_params  = 'real_params/'+'all_' + str(cosm_type)+'_params'+params_ext+'.txt' 

        # log        = True 
        # input_size = 79
        # model_type = 'dynamic_fixed_final'
        # final_hidden_layer_size = 10
        
    

        # INFO THAT DOES NOT CHANGE
        study_name = str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)
        mother = '/scratch/network/vk9342/USRP2024_scratch/pytorch/'+str(Pk_type)+'_'+str(cosm_type)+'/'+str(name)+'/models/'
        
        print(name)
        print(f_params)
        print(f_Pk)
        print(study_name)
        
        # architecture parameters
        # training parameters
        batch_size = 32
        # optuna parameters
        storage    = 'sqlite:///nwLH.db'
        
        ######################################################################################
        ## LOAD MODEL
        
        # use GPUs if available
        if torch.cuda.is_available():
            print("CUDA Available")
            device = torch.device('cuda')
        else:
            print('CUDA Not Available')
            device = torch.device('cpu')


        ################### OPTUNA BEST STUDY LOAD #################
        # BSQ_name = 'transfer10_network1_'+str(n_sims_BSQ)+'_BSQ_fixed'
        if cosm_type == 'fR':
            add_ext = '_fR'
        else:
            add_ext = ''
        
        BSQ_name = 'transfer10_network1_'+str(n_sims_BSQ)+'_BSQ' + additional_extension + add_ext
        BSQ_study_name = 'Pk_BSQ_params_'+BSQ_name
        BSQ_study = optuna.load_study(study_name=BSQ_study_name, storage=storage)
        values = np.zeros(len(BSQ_study.trials))
        completed = 0
        for i,t in enumerate(BSQ_study.trials):
            values[i] = t.value
            if t.value is not None:  completed += 1
        indexes = np.argsort(values)
        for i in [0]:  #choose the best-model here, e.g. [0], or [1]
            BSQ_best_trial = BSQ_study.trials[indexes[i]]
            BSQ_lr       = BSQ_best_trial.params['lr']
            BSQ_wd       = BSQ_best_trial.params['wd']
            BSQ_n_layers       = BSQ_best_trial.params['n_layers']
            BSQ_p       = BSQ_best_trial.params['dropout_l']
            if model_type == 'dynamic_fixed_final':
                BSQ_out_fs = [BSQ_best_trial.params[f'n_units_l{i}'] for i in range(BSQ_n_layers - 1)]
                BSQ_out_fhl = 10
            elif model_type == 'dynamic':
                BSQ_out_fs = [BSQ_best_trial.params[f'n_units_l{i}'] for i in range(BSQ_n_layers)]
            
        ########################################################################
        
        
        # load the optuna study
        study = optuna.load_study(study_name=study_name, storage=storage)
        # get the scores of the study trials
        values = np.zeros(len(study.trials))
        completed = 0
        for i,t in enumerate(study.trials):
            values[i] = t.value
            if t.value is not None:  completed += 1

        # get the info of the best trial            
        indexes = np.argsort(values)
        for indx in range(10):  #choose the best-model here, e.g. [0], or [1]
            trial = study.trials[indexes[indx]]
            print("\nTrial number {}".format(trial.number))
            print("Value: %.5e"%trial.value)
            print(" Params: ")
            for key, value in trial.params.items():
                print("    {}: {}".format(key, value))
            lr       = trial.params['lr']
            wd       = trial.params['wd']
            # n_layers       = trial.params['n_layers']
            n_layers = BSQ_n_layers
            p = BSQ_p
            out_fs = BSQ_out_fs
            # p       = trial.params['dropout_l']
            fmodel = mother +'model_%d.pt'%trial.number
        
            ## generate the architecture
            # out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers-1)]
           # generate architecture
            if model_type == 'dynamic_fixed_final':
                model = architecture.dynamic_model_fixed_final(trial, input_size, 
                                                               output_size, final_hidden_layer_size,
                                                               BSQ_n_layers, BSQ_p, BSQ_out_fs, max_neurons_layers=500
                                                              ).to(device)
            elif model_type == 'dynamic':
                model = architecture.dynamic_model(trial, input_size, 
                                                   output_size,
                                                   BSQ_n_layers, BSQ_p, BSQ_out_fs,
                                                   max_neurons_layers=500
                                                  ).to(device) 
    
            
            model.to(device)  
            # load best-model, if it exists
            if os.path.exists(fmodel):  
                print('Loading model...')
                model.load_state_dict(torch.load(fmodel, map_location=torch.device(device)))
            else:
                raise Exception('model doesnt exists!!!')
            
            # define loss function
            criterion = nn.MSELoss() 
    
    
            ## GET TRAIN, VALID, TEST RESULTS
            for mode in ['test']:
                # get the data
                if mode == 'train':
                        # get the data
                        full_train_dataset = data.create_dataset('train', seed, f_Pk, f_Pk_norm, 
                                                           f_params, batch_size, shuffle=False, 
                                                           workers=1, cosm_type = cosm_type, log=log, 
                                                           shuffle_all=True).dataset
                        train_subset, _ = torch.utils.data.random_split(full_train_dataset,
                                                                        [n_sims, len(full_train_dataset) - n_sims],
                                                                        generator=torch.Generator().manual_seed(seed)
                                                                       )
                        test_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=1)
                else:
                    test_loader = data.create_dataset(mode, seed, f_Pk, f_Pk_norm, f_params, 
                                                  batch_size, shuffle=False, workers=1, 
                                                  cosm_type =cosm_type, log=log, 
                                                  shuffle_all = True)
                
                    
                
                test_points = 0
                for x,y in test_loader:  test_points += x.shape[0]
                
                # define the arrays containing the true and predicted value of the parameters
                params  = output_size
                results = np.zeros((test_points, 2*params), dtype=np.float32)
                
                # test the model
                test_loss, points = 0.0, 0
                model.eval()
                with torch.no_grad():
                    for x, y in test_loader:
                        bs   = x.shape[0]  #batch size
                        x, y = x.to(device), y.to(device)
                        y_NN = model(x)
                        test_loss += (criterion(y_NN, y).item())*bs
                        results[points:points+bs,0*params:1*params] = y.cpu().numpy()
                        results[points:points+bs,1*params:2*params] = y_NN.cpu().numpy()
                        points    += bs
                test_loss /= points
                print('Test loss:', test_loss)
                
                # denormalize results here
                #
                #
                
                # save results to file
                fout = '../results/Results_'+mode+'_'+str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt'
                np.savetxt(fout, results)
                print('../results/Results_'+mode+'_'+str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt')
            
            
            
                    

transfer10_network2_50_nwLH_2000_BSQ_k_0.3
real_params/all_nwLH_params.txt
Pk_files/all_MPk_nwLH_k_0.3.npy
MPk_nwLH_params_transfer10_network2_50_nwLH_2000_BSQ_k_0.3
CUDA Available

Trial number 82
Value: 3.16102e-02
 Params: 
    lr: 0.0009933472542311417
    wd: 3.162092631133938e-05
Loading model...
Test loss: 0.03508919594188531
../results/Results_test_MPk_nwLH_params_transfer10_network2_50_nwLH_2000_BSQ_k_0.3_model0.txt

Trial number 26
Value: 3.16711e-02
 Params: 
    lr: 0.00048567035350112454
    wd: 3.851611444586862e-06
Loading model...
Test loss: 0.03582926707963149
../results/Results_test_MPk_nwLH_params_transfer10_network2_50_nwLH_2000_BSQ_k_0.3_model1.txt

Trial number 61
Value: 3.16753e-02
 Params: 
    lr: 0.000969785033547259
    wd: 8.700271794198282e-06
Loading model...
Test loss: 0.035471850012739496
../results/Results_test_MPk_nwLH_params_transfer10_network2_50_nwLH_2000_BSQ_k_0.3_model2.txt

Trial number 23
Value: 3.20444e-02
 Params: 
    lr: 0.000910139567332447

# Get top 10 models - BSQ nwLH combined with weights

In [12]:
################################### INPUT ############################################
# data parameters

f_Pk_norm = None #file with Pk to normalize Pk
seed      = 42       #seed to split data in train/valid/test
mode      = 'test'   #'train','valid','test' or 'all'

# params_ext = '_lcdm'

Pk_type = 'MPk'
cosm_type = 'nwLH'
cosm_type_data_range = 'nwLH_BSQ_combined'
k_max = 0.5

model_type = 'dynamic' #or 'dynamic' 'dynamic_fixed_final'
log = True

output_size = 6

if k_max == 0.5:
    input_size = 79
elif k_max == 0.3:
    input_size = 47
elif k_max == 0.1:
    input_size = 15

additional_extension =  '_combined_weighted'    #'_combined_train_val_revisions'   #FIXED TEST/VALID SET TO 300
params_ext = ''

n_sims_cosm = [50, 100, 200, 400, 800, 1400]
n_sims_BSQ = [22000]


In [13]:
for n_BSQ in n_sims_BSQ: 
    for n_sims in n_sims_cosm:
        n = []
        mse = []

        name = 'transfer10_network2_'+str(n_sims)+'_'+cosm_type+'_'+str(n_BSQ)+'_BSQ' + additional_extension

        f_Pk_cosm      = 'Pk_files/'+'all_'+str(Pk_type)+'_'+str(cosm_type)+params_ext+'.npy'
        f_params_cosm  = 'real_params/'+'all_' + str(cosm_type)+'_params'+params_ext+'.txt' 

        f_Pk_BSQ      = 'Pk_files/'+'all_Pk_BSQ.npy'
        f_params_BSQ  = 'real_params/'+'all_BSQ_params_6_col.txt' 
            
        

        study_name = str(Pk_type)+'_'+str(cosm_type)+'_params_'+str(name)
        mother = '/scratch/network/vk9342/USRP2024_scratch/pytorch/'+str(Pk_type)+'_'+str(cosm_type)+'/'+str(name)+'/models/'

        print(study_name)
        # print(f_params)
        # print(f_Pk)
        
    
        # architecture parameters
        # training parameters
        batch_size = 32
        # optuna parameters
        storage    = 'sqlite:///nwLH.db'
        
        
        
        ######################################################################################
        ######################################################################################
        # use GPUs if available
        if torch.cuda.is_available():
            print("CUDA Available")
            device = torch.device('cuda')
        else:
            print('CUDA Not Available')
            device = torch.device('cpu')
        
        # load the optuna study
        study = optuna.load_study(study_name=study_name, storage=storage)
        
        # get the scores of the study trials
        values = np.zeros(len(study.trials))
        completed = 0
        
        for i,t in enumerate(study.trials):
            values[i] = t.value
            if t.value is not None:  completed += 1
        
        # get the info of the best trial
        indexes = np.argsort(values)
        for indx in range(10):
            trial = study.trials[indexes[indx]]
        
            # print("\nTrial number {}".format(trial.number))
            # print("Value: %.5e"%trial.value)
            for key, value in trial.params.items():
                lr       = trial.params['lr']
                wd       = trial.params['wd']
                n_layers       = trial.params['n_layers']
                p       = trial.params['dropout_l']
                
            fmodel = mother +'model_%d.pt'%trial.number

            
        
            # # generate the architecture
            if model_type == 'dynamic_fixed_final':
                out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers-1)]
                model = architecture.dynamic_model_fixed_final(trial, input_size, output_size, 
                                                               final_hidden_layer_size, 
                                                               n_layers, p, out_fs,
                                                               max_neurons_layers=500)
                
            elif model_type == 'dynamic':
                out_fs = [trial.params[f'n_units_l{i}'] for i in range(n_layers)]
                model = architecture.dynamic_model(trial, input_size, output_size,
                                                   n_layers, p, out_fs,
                                                   max_neurons_layers=500)   

            else:
                print('not a valid model')

            # print('N BSQ: ', str(n_BSQ), 'N nwLH: ', str(n_nwLH), 'layer sizes: ', out_fs)
            # print('N BSQ: ', str(n_BSQ), 'N nwLH: ', str(n_nwLH), 'n layers: ', n_layers)
    
            
            model.to(device)  
            
            # load best-model, if it exists
            if os.path.exists(fmodel):  
                print('Loading model...')
                model.load_state_dict(torch.load(fmodel, map_location=torch.device(device)))
            else:
                raise Exception('model doesnt exists!!!')
            
            
            # define loss function
            criterion = nn.MSELoss() 
            
            # get the data
            if mode == 'test':
                test_loader = data.create_dataset(mode, seed, f_Pk_cosm, f_Pk_norm, f_params_cosm, 
                                                  batch_size, shuffle=False, workers=1, 
                                                      cosm_type =cosm_type_data_range, log=log, shuffle_all = True) 
            elif mode == 'valid':
                valid_loader_BSQ = data.create_dataset('valid', seed, f_Pk_BSQ, f_Pk_norm, 
                                           f_params_BSQ, batch_size, shuffle=False,
                                           workers=1, cosm_type = cosm_type_data_range, log=log, 
                                           shuffle_all = True)
                valid_loader_cosm = data.create_dataset('valid', seed, f_Pk_cosm, f_Pk_norm, 
                                           f_params_cosm, batch_size, shuffle=False,
                                           workers=1, cosm_type = cosm_type_data_range, log=log, 
                                           shuffle_all = True)
                
                combined_valid_dataset =torch.utils.data.ConcatDataset([valid_loader_BSQ.dataset, valid_loader_cosm.dataset])
                test_loader = DataLoader(combined_valid_dataset, batch_size=batch_size, shuffle=False, num_workers=1)
            
            elif mode == 'train':
                full_train_dataset_BSQ = data.create_dataset('train', seed, f_Pk_BSQ, f_Pk_norm, 
                                       f_params_BSQ, batch_size, shuffle=False, 
                                       workers=1, cosm_type = cosm_type_data_range, log=log, 
                                       shuffle_all=True).dataset
                
                train_subset_BSQ, _ = torch.utils.data.random_split(full_train_dataset_BSQ,
                                                                [n_BSQ, len(full_train_dataset_BSQ) - n_BSQ],
                                                                generator=torch.Generator().manual_seed(seed))
                
                full_train_dataset_cosm = data.create_dataset('train', seed, f_Pk_cosm, f_Pk_norm, 
                                                   f_params_cosm, batch_size, shuffle=False, 
                                                   workers=1, cosm_type = cosm_type_data_range, log=log, 
                                                   shuffle_all=True).dataset
                
                train_subset_cosm, _ = torch.utils.data.random_split(full_train_dataset_cosm,
                                                                [n_sims, len(full_train_dataset_cosm) - n_sims],
                                                                generator=torch.Generator().manual_seed(seed))
                combined_train_dataset = torch.utils.data.ConcatDataset([train_subset_BSQ, train_subset_cosm])
                w_lcdm = 1.0 / n_BSQ
                w_beyond = 1.0 / n_sims
                weights = torch.tensor([w_lcdm]*n_BSQ + [w_beyond]*n_sims)
        
                sampler = torch.utils.data.WeightedRandomSampler(
                    weights=weights, 
                    num_samples=len(weights), # Draws a full "epoch" worth of samples
                    replacement=True
                )
        
                test_loader = DataLoader(combined_train_dataset, batch_size=batch_size, 
                                          sampler=sampler, num_workers=1)  #shuffle = True
            else:
                print('mode??')
            
                                              
            test_points = 0
            for x,y in test_loader:  test_points += x.shape[0]
            
            # define the arrays containing the true and predicted value of the parameters
            params  = output_size
            results = np.zeros((test_points, 2*params), dtype=np.float32)
            
            # test the model
            test_loss, points = 0.0, 0
            model.eval()
            with torch.no_grad():
                for x, y in test_loader:
                    bs   = x.shape[0]  #batch size
                    x, y = x.to(device), y.to(device)
                    y_NN = model(x)
                    test_loss += (criterion(y_NN, y).item())*bs
                    results[points:points+bs,0*params:1*params] = y.cpu().numpy()
                    results[points:points+bs,1*params:2*params] = y_NN.cpu().numpy()
                    points    += bs
            test_loss /= points
            print('Test loss:', test_loss)
            
            fout = '../results/Results_'+mode+'_'+Pk_type+'_'+cosm_type+'_params_'+str(name)+params_ext+'_model'+str(indx)+'.txt'
            np.savetxt(fout, results)


MPk_nwLH_params_transfer10_network2_50_nwLH_22000_BSQ_combined_weighted
CUDA Available
Loading model...
Test loss: 0.039386434654394786
Loading model...
Test loss: 0.04126934324701627
Loading model...
Test loss: 0.042603616664807005
Loading model...
Test loss: 0.036508350123961766
Loading model...
Test loss: 0.041223709285259244
Loading model...
Test loss: 0.041684064666430154
Loading model...
Test loss: 0.03751912554105123
Loading model...
Test loss: 0.04079425846536954
Loading model...
Test loss: 0.042113781124353405
Loading model...
Test loss: 0.04461811160047849
MPk_nwLH_params_transfer10_network2_100_nwLH_22000_BSQ_combined_weighted
CUDA Available
Loading model...
Test loss: 0.02583974299331506
Loading model...
Test loss: 0.02593593602379163
Loading model...
Test loss: 0.027789745877186457
Loading model...
Test loss: 0.02755118434627851
Loading model...
Test loss: 0.028843189204732576
Loading model...
Test loss: 0.027817981392145155
Loading model...
Test loss: 0.027283099467555683